In [4]:
from scipy import stats
from statsmodels.stats.weightstats import ztest
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
import numpy as np
import statistics as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

data = pd.read_csv('nba_second_round_history.csv')
ts = pd.read_csv('nba_true_shooting_data.csv')
# hide 2026 season data since it's not complete and may skew the analysis
data = data[data['Season'] < 2026]
ts = ts[ts['Season'] < 2026]
# detect missing values
print('Missing values in data:', data.columns[data.isnull().any()].tolist() or 0)
print('Missing values in ts:', ts.columns[ts.isnull().any()].tolist() or 0)

data

Missing values in data: 0
Missing values in ts: 0


,Season,Team,Opponent,Games played,Average Points scored,Average Points allowed,Offensive Rating,Defensive Rating,eFG%,Opp eFG%,TOV%,Opp TOV%,ORB%,Opp ORB%,FT/FGA,Opp FT/FGA,Champion
0,1984,BOS,NYK,7,111.0,103.0,113.9,105.7,0.498,0.484,14.2,16.0,38.7,35.0,0.285,0.296,True
1,1984,MIL,NJN,6,98.2,96.3,102.8,100.9,0.471,0.405,18.3,12.2,32.6,35.5,0.373,0.340,False
2,1984,LAL,DAL,5,120.6,106.2,121.9,107.4,0.570,0.454,12.8,11.9,37.8,35.1,0.200,0.232,False
3,1984,PHO,UTA,6,103.7,101.0,107.0,104.3,0.462,0.468,12.9,15.3,33.9,32.4,0.230,0.300,False
4,1985,BOS,DET,6,120.5,112.7,118.1,110.4,0.519,0.478,13.6,11.4,37.1,35.7,0.302,0.196,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,2024,MIN,DEN,7,102.7,97.6,113.8,108.1,0.539,0.517,11.5,12.4,22.8,21.1,0.201,0.173,False
164,2025,IND,CLE,5,117.6,114.2,116.7,113.3,0.585,0.490,13.3,11.7,18.0,28.8,0.220,0.295,False
165,2025,NYK,BOS,6,105.7,105.2,112.9,112.4,0.508,0.515,11.7,12.2,29.6,27.6,0.211,0.176,False
166,2025,OKC,DEN,7,115.4,106.3,114.6,105.6,0.518,0.481,9.7,15.1,24.9,28.0,0.208,0.249,True


In [2]:
# 1. Add Net Rating column (淨效率值 = 每100回合的得分 - 每100回合的失分) (better measure of team performance than point differential because it accounts for pace)
data['Net Rating'] = (data['Offensive Rating'] - data['Defensive Rating']).round(3)

# 2. Add Average Points Difference column
data['Average Points Difference'] = (data['Average Points scored'] - data['Average Points allowed']).round(3)

# 3. Add eFG% Difference column (eFG% 有效命中率 = 總命中數＋0.5*三分球的命中數）/ 總出手數)
data['eFG% Difference'] = ((data['eFG%'] - data['Opp eFG%']) * 100).round(3)

# 4. Add TOV% Difference column (TOV% 失誤率 = 失誤數 / 總出手數)
data['TOV% Difference'] = ((data['TOV%'] - data['Opp TOV%'])).round(3)

# 5. Add ORB% Difference column
data['ORB% Difference'] = (data['ORB%'] - data['Opp ORB%']).round(3)

# 6. Add FT/FGA Difference column
data['FT/FGA Difference'] = (data['FT/FGA'] - data['Opp FT/FGA']).round(3)

# 7. Add Actual Wins column
data['Actual Wins (%)'] = (4 / data['Games played'] * 100).round(3)

# 8. Add Pythagorean Wins column (a measure of dominance)
data['Pythagorean Wins (13.91) (%)'] = ((data['Average Points scored'] ** 13.91) / ((data['Average Points scored'] ** 13.91) + (data['Average Points allowed'] ** 13.91)) * 100).round(3)
data['Pythagorean Wins (16.5) (%)'] = ((data['Average Points scored'] ** 16.5) / ((data['Average Points scored'] ** 16.5) + (data['Average Points allowed'] ** 16.5)) * 100).round(3)

# 9. Add True Shooting Percentage Difference column (TS% 真實命中率 = 總得分 / (2 * (總出手數 + 0.44 * 罰球數)))
data['TS% Difference'] = ((ts['Team TS%'] - ts['Opp TS%'])).round(3)

data.drop(columns=['Average Points scored', 'Average Points allowed', 'Offensive Rating', 'Defensive Rating', 'eFG%', 'Opp eFG%', 'TOV%', 'Opp TOV%', 'ORB%', 'Opp ORB%', 'FT/FGA', 'Opp FT/FGA'], inplace=True)
# Save the modified dataset to a new CSV file
data.to_csv('nba_second_round_history_advance.csv', index=False)

In [29]:
data = pd.read_csv('nba_second_round_history.csv')
data = data[data['Season'] < 2026]

data['Actual Wins (%)'] = (4 / data['Games played'] * 100).round(3)

data['Team TS%'] = ts['Team TS%']
data['Opp TS%'] = ts['Opp TS%']

data['eFG%'] = (data['eFG%'] * 100).round(1)
data['Opp eFG%'] = (data['Opp eFG%'] * 100).round(1)

data['FT/FGA'] = (data['FT/FGA'] * 100).round(1)
data['Opp FT/FGA'] = (data['Opp FT/FGA'] * 100).round(1)

data['DRB%'] = 100 - data['Opp ORB%']

data.drop(columns=['Opp ORB%'], inplace=True)

print(data.head())
data.to_csv('nba_second_round_history_augmented.csv', index=False)

   Season Team Opponent  Games played  Average Points scored  \
0    1984  BOS      NYK             7                  111.0   
1    1984  MIL      NJN             6                   98.2   
2    1984  LAL      DAL             5                  120.6   
3    1984  PHO      UTA             6                  103.7   
4    1985  BOS      DET             6                  120.5   

   Average Points allowed  Offensive Rating  Defensive Rating  eFG%  Opp eFG%  \
0                   103.0             113.9             105.7  49.8      48.4   
1                    96.3             102.8             100.9  47.1      40.5   
2                   106.2             121.9             107.4  57.0      45.4   
3                   101.0             107.0             104.3  46.2      46.8   
4                   112.7             118.1             110.4  51.9      47.8   

   TOV%  Opp TOV%  ORB%  FT/FGA  Opp FT/FGA  Champion  Actual Wins (%)  \
0  14.2      16.0  38.7    28.5        29.6      True 

Standardise data (Use the **regular season** avg data of the 16 playoff teams)

In [ ]:
# preprocess
playoff_teams = pd.read_csv('nba_playoff_teams_RAW_1984_2026.csv')
playoff_teams = playoff_teams[playoff_teams['Season'] < 2026]

playoff_teams.rename({'PTS': 'Average Points scored', 'OPP_PTS': 'Average Points allowed', 'ORtg': 'Offensive Rating', \
    'DRtg': 'Defensive Rating'}, axis=1, inplace=True)

for col in playoff_teams.columns:
    if playoff_teams[col].dtype == 'float64' and playoff_teams[col].max() <= 1:
        playoff_teams[col] = (playoff_teams[col] * 100).round(3)
        
data = pd.read_csv('nba_second_round_history_augmented.csv')

,Season,Team,Opponent,Games played,Average Points scored,Average Points allowed,Offensive Rating,Defensive Rating,eFG%,Opp eFG%,TOV%,Opp TOV%,ORB%,FT/FGA,Opp FT/FGA,Champion,Actual Wins (%),Team TS%,Opp TS%,DRB%
0,1984,BOS,NYK,7,111.0,103.0,113.9,105.7,49.8,48.4,14.2,16.0,38.7,28.5,29.6,True,57.143,54.9350,53.9574,65.0
1,1984,MIL,NJN,6,98.2,96.3,102.8,100.9,47.1,40.5,18.3,12.2,32.6,37.3,34.0,False,66.667,54.0129,47.1452,64.5
2,1984,LAL,DAL,5,120.6,106.2,121.9,107.4,57.0,45.4,12.8,11.9,37.8,20.0,23.2,False,80.000,59.7550,50.4063,64.9
3,1984,PHO,UTA,6,103.7,101.0,107.0,104.3,46.2,46.8,12.9,15.3,33.9,23.0,30.0,False,66.667,51.2187,52.6627,67.6
4,1985,BOS,DET,6,120.5,112.7,118.1,110.4,51.9,47.8,13.6,11.4,37.1,30.2,19.6,False,66.667,57.8882,51.6914,64.3


In [32]:
# 1. Identify the numeric columns present in BOTH DataFrames
stats_to_standardize = [
    stat for stat in playoff_teams.columns 
    if stat in data.columns and data[stat].dtype in [np.float64, np.int64]
]

# 2. Calculate baseline mean and std for ALL stats at once per season
# This completely avoids the merge-in-a-loop issue and is much faster
baseline_df = playoff_teams.groupby('Season')[stats_to_standardize].agg(['mean', 'std'])

# Flatten the MultiIndex columns (e.g., ('PTS', 'mean') becomes 'PTS_mean')
baseline_df.columns = [f'{col[0]}_{col[1]}' for col in baseline_df.columns]
baseline_df = baseline_df.reset_index() # Turn 'Season' index back into a regular column

# 3. Merge the baseline stats with the main data ONCE
# This automatically broadcasts the 1 row per season baseline to the 4 rows per season data
merged_df = pd.merge(data, baseline_df, on='Season', how='left')

# 4. Initialize the final DataFrame with the non-standardized columns
standard_df = data[['Season', 'Champion', 'Games played']].copy()

# 5. Loop through and calculate the Z-scores
for stat in stats_to_standardize:
    mean_col = f'{stat}_mean'
    std_col = f'{stat}_std'
    
    # Z-score formula: (value - mean) / std
    # Note: If a season has a std of 0 (unlikely but possible), this will return NaN, which is mathematically correct.
    standard_df[stat] = ((merged_df[stat] - merged_df[mean_col]) / merged_df[std_col]).round(3)

standard_df['Season'] = data['Season']
# 6. Save to CSV
standard_df.to_csv('nba_second_round_history_standardised.csv', index=False)